In [42]:
import pandas as pd
import numpy as np

In [43]:
df = pd.read_json(
    "/content/Sarcasm_Headlines_Dataset.json",
    lines=True
)

df.head()

,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1
3,https://politics.theonion.com/boehner-just-wan...,"boehner just wants wife to listen, not come up...",1
4,https://www.huffingtonpost.com/entry/jk-rowlin...,j.k. rowling wishes snape happy birthday in th...,0


In [44]:
df.shape

(26709, 3)

In [45]:
df.isnull().sum()

,0
article_link,0
headline,0
is_sarcastic,0


In [46]:
df.duplicated().sum()

np.int64(1)

In [47]:
df = df.drop_duplicates()

In [48]:
df = df[['headline', 'is_sarcastic']]

In [49]:
df.head()

,headline,is_sarcastic
0,former versace store clerk sues over secret 'b...,0
1,the 'roseanne' revival catches up to our thorn...,0
2,mom starting to fear son's web series closest ...,1
3,"boehner just wants wife to listen, not come up...",1
4,j.k. rowling wishes snape happy birthday in th...,0


In [50]:
df['headline'] = df['headline'].astype(str)

In [51]:
X = df['headline']
y = df['is_sarcastic']

print(X.shape)
print(y.shape)

(26708,)
(26708,)


In [52]:
import sklearn
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [53]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense,Input

In [54]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")

tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print(X_train_seq[:2])

[[4463, 120, 1, 1706, 3994, 273, 21, 558, 1798, 306], [84, 695, 18, 100, 23, 470, 43, 1, 112]]


In [55]:
from keras.preprocessing.sequence import pad_sequences

In [56]:
max_len = 200
X_train_seq = pad_sequences(X_train_seq,maxlen=max_len,padding='post',truncating='post')
X_test_seq = pad_sequences(X_test_seq,maxlen=max_len,padding='post',truncating='post')

In [57]:

np.unique(y_train)

array([0, 1])

In [58]:
voc_size = 5000
model_rnn = Sequential([Input(shape=(max_len,)),
                        Embedding(input_dim=voc_size,output_dim=32),
                        SimpleRNN(64,activation='tanh'),
                        Dense(1,activation='sigmoid')
                        ])

In [59]:

model_rnn.compile(loss="binary_crossentropy",
                  optimizer='adam',
                  metrics=['accuracy'])

In [63]:
history_rnn = model_rnn.fit(X_train_seq,y_train,
                            epochs=4,
                            validation_split = 0.2,
                            batch_size = 64)

Epoch 1/4
268/268 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5626 - loss: 0.6866 - val_accuracy: 0.5547 - val_loss: 0.6874
Epoch 2/4
268/268 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.5532 - loss: 0.6891 - val_accuracy: 0.5190 - val_loss: 0.6903
Epoch 3/4
268/268 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.5677 - loss: 0.6755 - val_accuracy: 0.5746 - val_loss: 0.6731
Epoch 4/4
268/268 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.5816 - loss: 0.6695 - val_accuracy: 0.5686 - val_loss: 0.6740


In [64]:
loss , acc = model_rnn.evaluate(X_test_seq,y_test)

167/167 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5758 - loss: 0.6678


In [65]:
print("Test Loss:", loss)
print("Test Accuracy:", acc)
print("Test Accuracy (%):", acc * 100)

Test Loss: 0.6678133606910706
Test Accuracy: 0.5758143067359924
Test Accuracy (%): 57.58143067359924
